In [3]:
# 01 – Data Collection

# This notebook collects Google Play Store reviews for four budgeting apps: **Credit Karma, Rocket Money, YNAB, and Monarch Money**. 
# The data covers June 2023 to today, which includes the period before and after Mint shut down in March 2024.

# To do this, I will: 
# 1. Set up the app list
# 2. Run a small test scrape to confirm the app IDs and data fields
# 3. Scrape all reviews from June 2023 onward
# 4. Save the raw data and check whether it can answer the project's business questions

In [9]:

from google_play_scraper import reviews, Sort
import pandas as pd

apps = {
    "Credit Karma": "com.creditkarma.mobile",
    "Rocket Money": "com.truebill",
    "YNAB": "com.youneedabudget.evergreen.app",
    "Monarch Money": "com.monarchmoney.mobile",
}

In [2]:
%pip install google-play-scraper pandas

  Obtaining dependency information for google-play-scraper from https://files.pythonhosted.org/packages/33/f7/a23ef3cf8efc9ab3aee565971f59906811e6ce95475314ef7b18d02f30ba/google_play_scraper-1.2.7-py3-none-any.whl.metadata
  Using cached google_play_scraper-1.2.7-py3-none-any.whl.metadata (50 kB)
Using cached google_play_scraper-1.2.7-py3-none-any.whl (28 kB)
Note: you may need to restart the kernel to use updated packages.


In [4]:
# testing if data scraping works
test_data = []

for name, app_id in apps.items():
    result, _ = reviews(app_id, lang="en", country="us", sort=Sort.NEWEST, count=200)
    for r in result:
        r["app_name"] = name
    test_data.extend(result)
    print(f"{name}: {len(result)} reviews")

Credit Karma: 200 reviews
Rocket Money: 200 reviews
YNAB: 200 reviews
Monarch Money: 200 reviews


In [5]:
# validate result of data testing
test_df = pd.DataFrame(test_data)
print(test_df.columns.tolist())
test_df[["app_name", "score", "at", "content"]].head(10)

['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'app_name']


,app_name,score,at,content
0,Credit Karma,1,2026-09-22 19:12:51,I've submitted several selfies and pictures of...
1,Credit Karma,5,2026-09-22 19:02:20,it's all about karma!
2,Credit Karma,5,2026-09-22 17:48:53,Very up-to-date with your credit. Definitely a...
3,Credit Karma,3,2026-09-22 17:30:32,watch for errors in creditkarma reports
4,Credit Karma,5,2026-09-22 13:29:58,like the holistic view into my finances
5,Credit Karma,4,2026-09-22 13:06:40,Very Informative. I love the app.
6,Credit Karma,5,2026-09-22 12:40:08,Excellent way to manage your credit activity.
7,Credit Karma,1,2026-09-22 11:58:39,Mint was a lot better
8,Credit Karma,1,2026-09-22 11:32:57,credit karma is a fraud. its wrong advertiseme...
9,Credit Karma,1,2026-09-22 10:20:32,I don't want this app.but because I am paying ...


In [6]:
# real full scrape back to mid-2023
from datetime import datetime
import time

START_DATE = datetime(2023, 6, 1)

def scrape_app(name, app_id):
    all_reviews = []
    token = None
    while True:
        batch, token = reviews(app_id, lang="en", country="us", sort=Sort.NEWEST,
                               count=200, continuation_token=token)
        all_reviews.extend(batch)
        print(f"{name}: {len(all_reviews)} collected so far")
        
         # stop if there are no more reviews, or the oldest one here is before June 2023
        if not batch or batch[-1]["at"] < START_DATE:
            break
        time.sleep(1)
    return [r for r in all_reviews if r["at"] >= START_DATE]



In [7]:
# scrape all review data
frames = []
for name, app_id in apps.items():
    df = pd.DataFrame(scrape_app(name, app_id))
    df["app_name"] = name
    df = df.drop(columns=["userName", "userImage"])
    filename = name.lower().replace(" ", "_")
    df.to_csv(f"../data/raw/{filename}_reviews.csv", index=False)
    frames.append(df)
    print(f"DONE {name}: {len(df)} reviews saved\n")

Credit Karma: 200 collected so far
Credit Karma: 400 collected so far
Credit Karma: 600 collected so far
Credit Karma: 800 collected so far
Credit Karma: 1000 collected so far
Credit Karma: 1200 collected so far
Credit Karma: 1400 collected so far
Credit Karma: 1600 collected so far
Credit Karma: 1800 collected so far
Credit Karma: 2000 collected so far
Credit Karma: 2200 collected so far
Credit Karma: 2400 collected so far
Credit Karma: 2600 collected so far
Credit Karma: 2800 collected so far
Credit Karma: 3000 collected so far
Credit Karma: 3200 collected so far
Credit Karma: 3400 collected so far
Credit Karma: 3600 collected so far
Credit Karma: 3800 collected so far
Credit Karma: 4000 collected so far
Credit Karma: 4200 collected so far
Credit Karma: 4400 collected so far
Credit Karma: 4600 collected so far
Credit Karma: 4800 collected so far
Credit Karma: 5000 collected so far
Credit Karma: 5200 collected so far
Credit Karma: 5400 collected so far
Credit Karma: 5600 collected so 

Rocket Money: 13600 collected so far
Rocket Money: 13800 collected so far
Rocket Money: 14000 collected so far
Rocket Money: 14200 collected so far
Rocket Money: 14400 collected so far
Rocket Money: 14600 collected so far
Rocket Money: 14800 collected so far
Rocket Money: 15000 collected so far
Rocket Money: 15200 collected so far
Rocket Money: 15400 collected so far
Rocket Money: 15600 collected so far
Rocket Money: 15800 collected so far
Rocket Money: 16000 collected so far
Rocket Money: 16200 collected so far
Rocket Money: 16400 collected so far
Rocket Money: 16600 collected so far
Rocket Money: 16800 collected so far
Rocket Money: 17000 collected so far
Rocket Money: 17200 collected so far
Rocket Money: 17400 collected so far
Rocket Money: 17600 collected so far
Rocket Money: 17800 collected so far
Rocket Money: 18000 collected so far
Rocket Money: 18200 collected so far
Rocket Money: 18400 collected so far
Rocket Money: 18600 collected so far
Rocket Money: 18800 collected so far
R

In [8]:
# check data
all_df = pd.concat(frames) # combines all apps into one table
all_df["mentions_mint"] = all_df["content"].str.contains(r"\bmint\b", case=False, na=False) # mark T/F if mention Mint or not

print("Total reviews per app:\n", all_df.groupby("app_name").size(), "\n")
print("Mint mentions per app:\n", all_df.groupby("app_name")["mentions_mint"].sum(), "\n")

ck = all_df[all_df["app_name"] == "Credit Karma"]
budget_words = r"budget|spending|transaction|categor|net worth|mint"
print("Credit Karma budgeting reviews:", ck["content"].str.contains(budget_words, case=False, na=False).sum())
print("Share missing app version:", round(all_df["reviewCreatedVersion"].isna().mean(), 2))


Total reviews per app:
 app_name
Credit Karma     31390
Monarch Money     2443
Rocket Money     22596
YNAB              2064
dtype: int64 

Mint mentions per app:
 app_name
Credit Karma     2792
Monarch Money     441
Rocket Money      229
YNAB               35
Name: mentions_mint, dtype: int64 

Credit Karma budgeting reviews: 3130
Share missing app version: 0.11


In [15]:
pd.set_option("display.max_colwidth", None)  # show full review text instead of cutting it off

# pick 8 reviews for every app and every star rating (160 reviews total)
sample = all_df.groupby(["app_name", "score"]).sample(n=8, random_state=42)

# save them as a file you can open like a spreadsheet
sample[["app_name", "score", "content"]].to_csv("../data/review_sample.csv", index=False)



In [11]:
sample[["app_name", "score", "content"]].to_csv("../data/review_sample.csv", index=False)  # save sample for reading